# Session 18 Exercise: Regularization And Pipelines

Tune a ridge regression model inside a preprocessing pipeline.


In [ ]:
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import Ridge
from sklearn.model_selection import GridSearchCV, KFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

from pathlib import Path


def find_repo_root(start=Path.cwd()):
    for path in [start, *start.parents]:
        if (path / "pyproject.toml").exists():
            return path
    raise RuntimeError("Could not find repository root")


ROOT = find_repo_root()
DATA = ROOT / "data" / "raw"


In [ ]:
housing = pd.read_csv(DATA / "housing_sales.csv")
features = ["size_sq_m", "rooms", "age_years", "renovation_score", "near_transit", "district"]
X = housing[features]
y = housing["price_k_eur"]
numeric = ["size_sq_m", "rooms", "age_years", "renovation_score", "near_transit"]
preprocess = ColumnTransformer(
    [
        ("num", StandardScaler(), numeric),
        ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), ["district"]),
    ]
)
pipe = Pipeline([("preprocess", preprocess), ("model", Ridge())])


In [ ]:
grid = GridSearchCV(
    pipe,
    param_grid={"model__alpha": [0.1, 1.0, 10.0, 100.0]},
    cv=KFold(n_splits=5, shuffle=True, random_state=42),
    scoring="neg_root_mean_squared_error",
)
grid.fit(X, y)
print(grid.best_params_)
print(f"Best CV RMSE: {-grid.best_score_:.2f}")


Add one larger alpha value. What happens to the selected model and cross-validated error?
